# Lab 05 solution, notebook 2: a fourth skill, and how much of the choice the name carries

The worked answer to the challenge of `05b_context_and_skills_at_the_desk.ipynb`. The setup repeats the notebook's
skill functions, lookups, tools and loop, so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import json
import os
import sqlite3
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from openai import OpenAI, omit

client = OpenAI(timeout=60, max_retries=2)

`read_skills` reads every `SKILL.md` under a folder into a dictionary, as Part A did for `skills/`. `SKILLS` is the
dictionary that `load_skill` and `read_skill_file` look a skill up in; the tasks below replace it.

In [3]:
import yaml

LAB = ROOT / "05_Context_And_Skills"


def read_skills(folder):
    # Every SKILL.md under a folder: its description, its body and where it lives
    skills = {}
    for path in sorted(folder.glob("*/SKILL.md")):
        before, frontmatter, body = path.read_text().split("---", 2)
        meta = yaml.safe_load(frontmatter)
        skills[meta["name"]] = {"description": meta["description"], "body": body.strip(), "folder": path.parent}
    return skills


SKILLS = read_skills(LAB / "skills")
print(list(SKILLS))

['account-changes', 'damaged-media', 'refund-requests']


In [4]:
def load_skill(name):
    # Level 2: the body of one SKILL.md
    return SKILLS[name]["body"]


def read_skill_file(skill, path):
    # Level 3: one file inside the skill's folder, and nothing outside it
    folder = SKILLS[skill]["folder"].resolve()
    target = (folder / path).resolve()
    if folder not in target.parents or not target.is_file():
        return "No such file in this skill."
    return target.read_text()

In [5]:
DB_PATH = ROOT / "data" / "chinook.db"
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True, check_same_thread=False)


def find_customer(name: str) -> list[dict]:
    """Find a shop customer by full name. Returns a list of matches with their customer_id.

    Args:
        name: First and last name, for example 'Ana Silva'.
    """
    rows = conn.execute(
        "SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
        "WHERE FirstName || ' ' || LastName = ?", (name,)).fetchall()
    return [{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows]


def get_invoices(customer_id: int) -> dict:
    """Count a customer's invoices and add up their total in US dollars."""
    count, total = conn.execute(
        "SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


def get_support_rep(customer_id: int) -> dict:
    """Name the support rep who looks after a customer."""
    row = conn.execute(
        "SELECT e.FirstName || ' ' || e.LastName FROM customers c "
        "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "support_rep": row[0] if row else None}

In [6]:
lookup_tools = [
    {
        "type": "function",
        "function": {
            "name": "find_customer",
            "description": "Find a shop customer by full name. Returns a list of matches with their customer_id.",
            "parameters": {
                "type": "object",
                "properties": {"name": {"type": "string", "description": "First and last name"}},
                "required": ["name"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_invoices",
            "description": "Count a customer's invoices and add up their total in US dollars.",
            "parameters": {
                "type": "object",
                "properties": {"customer_id": {"type": "integer"}},
                "required": ["customer_id"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_support_rep",
            "description": "Name the support rep who looks after a customer.",
            "parameters": {
                "type": "object",
                "properties": {"customer_id": {"type": "integer"}},
                "required": ["customer_id"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    },
]

In [7]:
FUNCTIONS = {
    "load_skill": load_skill, "read_skill_file": read_skill_file,
    "find_customer": find_customer, "get_invoices": get_invoices, "get_support_rep": get_support_rep,
}


def run_calls(reply, messages):
    # Add the model's request, then run each tool it asked for and add each result
    messages.append(reply)
    for call in reply.tool_calls:
        arguments = json.loads(call.function.arguments)
        print("   the model asked for:", call.function.name, arguments)
        result = FUNCTIONS[call.function.name](**arguments)
        if not isinstance(result, str):
            result = json.dumps(result, ensure_ascii=False)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

In [8]:
def ask(question, system, tools, max_turns=8):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": question}]
    tokens = []

    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools, reasoning_effort=EFFORT or omit
        )
        reply = response.choices[0].message
        tokens.append(response.usage.prompt_tokens)
        print(f"call {turn + 1}: {response.usage.prompt_tokens} input tokens")

        if not reply.tool_calls:
            return reply.content, tokens
        run_calls(reply, messages)

    return "Stopped: the model was still asking for tools after max_turns calls.", tokens

The skill tools and the thin prompt depend on which skills exist and what the list says about them, so `build`
makes both from a dictionary of skills.

In [9]:
BASE = (
    "You help the support reps of the Chinook music shop. Look up every customer fact with the tools; "
    "never answer from memory. Answer in two or three sentences."
)


def build(skills):
    # The five tool schemas and the thin system prompt for a set of skills
    skill_tools = [
        {
            "type": "function",
            "function": {
                "name": "load_skill",
                "description": "Load the full steps of one skill from the skill list.",
                "parameters": {
                    "type": "object",
                    "properties": {"name": {"type": "string", "enum": list(skills)}},
                    "required": ["name"],
                    "additionalProperties": False,
                },
                "strict": True,
            },
        },
        {
            "type": "function",
            "function": {
                "name": "read_skill_file",
                "description": "Read a file inside a skill's folder, when the skill's steps point to it.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "skill": {"type": "string", "enum": list(skills)},
                        "path": {"type": "string", "description": "Path inside the skill folder"},
                    },
                    "required": ["skill", "path"],
                    "additionalProperties": False,
                },
                "strict": True,
            },
        },
    ]
    skill_list = "\n".join(f"- {name}: {skill['description']}" for name, skill in skills.items())
    thin = (
        BASE
        + " The shop's rules are kept as skills. Each skill below holds the steps for one kind of request. "
        + "Before you answer a question about the shop's rules, load the skill that covers it with load_skill, "
        + "and read a file inside it with read_skill_file when its steps tell you to. "
        + "If no skill covers the question, say that the rules you have do not cover it.\n\n"
        + "Skills:\n"
        + skill_list
    )
    return lookup_tools + skill_tools, thin

## Task 1: write a fourth skill

Our `digital-downloads` skill is in `solved/my_skills/digital-downloads/SKILL.md`. Yours goes in
`my_skills/digital-downloads/` beside `05b_context_and_skills_at_the_desk.ipynb`, and is added the same way. The
body restates `data/corpus/policy_digital_downloads.md` as steps, and the description says what the skill covers and
when to use it.

In [10]:
MY_SKILLS = LAB / "solved" / "my_skills"

print((MY_SKILLS / "digital-downloads" / "SKILL.md").read_text())

---
name: digital-downloads
description: Re-issue or fix the download of a digital album, when the link has expired, the file is corrupt or the customer asks to send it to another email. Use when a customer cannot download an album they bought, or a downloaded file will not open.
---

# Digital downloads

1. **An expired link.** A download link lasts 90 days after purchase. The customer can ask for one new link, and it
   lasts 7 days. Say so, and send the new link.
2. **Another email.** Downloads are tied to the email on the invoice. We do not send a second copy to a different
   email; for a new email on the account, use the `account-changes` skill.
3. **A corrupt file.** There is no return and no refund on a completed download, unless the file is corrupt. Then
   re-issue the link. A failed download is urgent: it jumps the support queue.

Your reply says what we send, how long it lasts, and what we cannot do.



In [11]:
SKILLS.update(read_skills(MY_SKILLS))
ALL_TOOLS, THIN = build(SKILLS)

print(list(SKILLS))

['account-changes', 'damaged-media', 'refund-requests', 'digital-downloads']


In [12]:
LINK = "A customer bought a digital album four months ago and the download link no longer works. What can we do?"

answer, tokens = ask(LINK, THIN, ALL_TOOLS)
print(answer)

call 1: 664 input tokens
   the model asked for: load_skill {'name': 'digital-downloads'}


call 2: 856 input tokens
Download links expire after 90 days, so we can issue one new link that will work for 7 days. Downloads are tied to the email on the invoice, so we can’t send a second copy to a different email.


In [13]:
HELENA = "How many invoices does Helena Holý have, what do they add up to, and who is her support rep?"

answer, tokens = ask(HELENA, THIN, ALL_TOOLS)
print(answer)

call 1: 665 input tokens
   the model asked for: find_customer {'name': 'Helena Holý'}


call 2: 722 input tokens
   the model asked for: get_invoices {'customer_id': 6}
   the model asked for: get_support_rep {'customer_id': 6}


call 3: 825 input tokens
Helena Holý has 7 invoices totaling $49.62. Her support rep is Steve Johnson.


The download question loaded `digital-downloads` and nothing else, in two calls, and the answer has the rule from our skill: one new link, which lasts 7 days. Helena's question made the same three lookups as in the notebook and loaded no skill.

**What the fourth skill costs a question that does not need it:** Helena's first call sent 665 input tokens here, against 596 in the notebook with three skills. One more line in the list, on every call.

## Task 2: find out how much of the choice the name carries

### The real names, with descriptions that say nothing

We keep the three skills of the lab and replace each description with *Shop rules, part N*.
`{**skill, "description": ...}` makes a copy of a skill's dictionary with one value replaced.

Our prediction: the model still loads `refund-requests` for the duplicate charge, because the name alone says
what the skill is about.

In [14]:
DUPLICATE = "A customer was charged twice for the same album on the same day. What do we do?"

real = read_skills(LAB / "skills")

SKILLS = {
    name: {**skill, "description": f"Shop rules, part {number}"}
    for number, (name, skill) in enumerate(real.items(), start=1)
}
ALL_TOOLS, THIN = build(SKILLS)
print(THIN[THIN.index("Skills:"):])
print()

answer, tokens = ask(DUPLICATE, THIN, ALL_TOOLS)
print(answer)

Skills:
- account-changes: Shop rules, part 1
- damaged-media: Shop rules, part 2
- refund-requests: Shop rules, part 3



call 1: 467 input tokens
   the model asked for: load_skill {'name': 'refund-requests'}


call 2: 718 input tokens
   the model asked for: read_skill_file {'skill': 'refund-requests', 'path': 'references/duplicate-charges.md'}


call 3: 971 input tokens
For two invoices on the same day for the same album, refund the full amount on the later invoice (the one with the higher invoice number) and include both invoice numbers with the request. The refund goes to the original payment method 5–10 business days after approval; support decides within 3 business days of your first reply. If the invoice amounts differ, escalate it instead.


The list now says nothing about any skill except its name, and the model still went straight to `refund-requests`, then to the reference file, and answered in three calls. As predicted: a name that says what the skill is about does much of the description's work.

**What this does not show:** that descriptions are optional. The name `refund-requests` happens to contain the word the question is about. A request that does not use the skill's own words (*she wants her money back*, *the same album appears twice on her card*) has only the description to match against.

### Names that say nothing either

Now the names go too: `rules-1`, `rules-2` and `rules-3`. The dictionary's keys are the new names, so `load_skill`
finds each skill under its label.

Our prediction: with nothing to choose by, the model opens skills until it finds the one about refunds.

In [15]:
SKILLS = {
    f"rules-{number}": {**skill, "description": f"Shop rules, part {number}"}
    for number, skill in enumerate(real.values(), start=1)
}
ALL_TOOLS, THIN = build(SKILLS)
print(THIN[THIN.index("Skills:"):])
print()

answer, tokens = ask(DUPLICATE, THIN, ALL_TOOLS)
print(answer)

Skills:
- rules-1: Shop rules, part 1
- rules-2: Shop rules, part 2
- rules-3: Shop rules, part 3



call 1: 468 input tokens
   the model asked for: load_skill {'name': 'rules-1'}


call 2: 651 input tokens
   the model asked for: load_skill {'name': 'rules-2'}


call 3: 847 input tokens
   the model asked for: load_skill {'name': 'rules-3'}


call 4: 1098 input tokens
   the model asked for: read_skill_file {'skill': 'rules-3', 'path': 'references/duplicate-charges.md'}


call 5: 1351 input tokens
Check both invoice numbers and amounts; if the amounts differ, escalate because it isn’t treated as a duplicate. If they match, request a full refund for the later invoice (the higher invoice number), attaching both invoice numbers; the refund goes to the original payment method within 5–10 business days after approval, and support decides within 3 business days of your first reply.


Five calls instead of three. The model loaded `rules-1`, then `rules-2`, then `rules-3`, one per call, and only the third was the refund skill. Then it read the reference and answered. The last call sent 1,351 input tokens, against 971 with the real names, because the two skills it did not need are now in the conversation.

The answer is still right. As with the house-sitter's numbered notes in notebook 1, nothing in the reply shows the wasted loads; only the list of calls does.

**Lesson to Remember**

| The list says | What the model loaded for the duplicate charge | Calls |
|---|---|---|
| a clear name and a clear description | `refund-requests`, then its reference | 3 |
| a clear name, *Shop rules, part N* | `refund-requests`, then its reference | 3 |
| `rules-N`, *Shop rules, part N* | all three skills, then the reference | 5 |

The name and the description are both read at level 1, and together they are all the model has to choose by. The format asks for a folder name that matches the skill's `name`, so naming the folder well is the first half of writing the description.